<a href="https://colab.research.google.com/github/ishika240047/CS.23.204/blob/main/raksha1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()

Saving raksha-data-set.xlsx to raksha-data-set.xlsx


In [2]:
import pandas as pd
df = pd.read_excel("raksha-data-set.xlsx")
df.head()

,Record_ID,Date,Location,Crop_Type,Temperature_C,Spoilage_Observed,Storage_Available,Humidity_Percent,Days_After_Harvest,Storage_Temperature_C,Exposure_To_Sun_After_Harvest,Packaging_Type,Crop_Shelf_Life_Days
0,1,2025-02-02,Village_C,Tomato,32.6,Yes,Yes,67,10,13.8,6,Sack,7
1,2,2025-02-03,Village_A,Onion,27.2,Yes,Yes,76,5,14.7,3,Sack,60
2,3,2025-02-04,Village_C,Onion,21.1,Yes,No,83,11,21.0,5,Open,60
3,4,2025-02-05,Village_B,Wheat,34.9,No,Yes,39,1,15.5,3,Cold_Box,180
4,5,2025-02-06,Village_C,Potato,32.5,Yes,No,79,7,30.8,5,Open,30


In [3]:
df.columns

Index(['Record_ID', 'Date', 'Location', 'Crop_Type', 'Temperature_C',
       'Spoilage_Observed', 'Storage_Available', 'Humidity_Percent',
       'Days_After_Harvest', 'Storage_Temperature_C',
       'Exposure_To_Sun_After_Harvest', 'Packaging_Type',
       'Crop_Shelf_Life_Days'],
      dtype='object')

In [4]:
df.head()

,Record_ID,Date,Location,Crop_Type,Temperature_C,Spoilage_Observed,Storage_Available,Humidity_Percent,Days_After_Harvest,Storage_Temperature_C,Exposure_To_Sun_After_Harvest,Packaging_Type,Crop_Shelf_Life_Days
0,1,2025-02-02,Village_C,Tomato,32.6,Yes,Yes,67,10,13.8,6,Sack,7
1,2,2025-02-03,Village_A,Onion,27.2,Yes,Yes,76,5,14.7,3,Sack,60
2,3,2025-02-04,Village_C,Onion,21.1,Yes,No,83,11,21.0,5,Open,60
3,4,2025-02-05,Village_B,Wheat,34.9,No,Yes,39,1,15.5,3,Cold_Box,180
4,5,2025-02-06,Village_C,Potato,32.5,Yes,No,79,7,30.8,5,Open,30


In [6]:
df["Spoilage_Observed"] = df["Spoilage_Observed"].map({"Yes" : 1, "No" : 0})

### Feature and Target Selection

I will now separate the features (input variables) from the target variable (`Spoilage_Observed`).

Features to be considered include `Location`, `Crop_Type`, `Temperature_C`, `Storage_Available`, `Humidity_Percent`, `Days_After_Harvest`, `Storage_Temperature_C`, `Exposure_To_Sun_After_Harvest`, `Packaging_Type`, and `Crop_Shelf_Life_Days`.

`Record_ID` is an identifier and `Date` is a timestamp, which are typically not used directly as features for a classification model and will be excluded for now.

In [7]:
# Define the target variable (y)
y = df['Spoilage_Observed']

# Define the feature variables (X)
X = df.drop(columns=['Record_ID', 'Date', 'Spoilage_Observed'])

print("Features (X) head:")
display(X.head())

print("\nTarget (y) head:")
display(y.head())

Features (X) head:


,Location,Crop_Type,Temperature_C,Storage_Available,Humidity_Percent,Days_After_Harvest,Storage_Temperature_C,Exposure_To_Sun_After_Harvest,Packaging_Type,Crop_Shelf_Life_Days
0,Village_C,Tomato,32.6,Yes,67,10,13.8,6,Sack,7
1,Village_A,Onion,27.2,Yes,76,5,14.7,3,Sack,60
2,Village_C,Onion,21.1,No,83,11,21.0,5,Open,60
3,Village_B,Wheat,34.9,Yes,39,1,15.5,3,Cold_Box,180
4,Village_C,Potato,32.5,No,79,7,30.8,5,Open,30



Target (y) head:


,Spoilage_Observed
0,1
1,1
2,1
3,0
4,1


### Split Data into Training and Testing Sets

To evaluate the performance of our machine learning model, we need to split the data into two sets:

1.  **Training Set**: Used to train the model.
2.  **Testing Set**: Used to evaluate the model's performance on unseen data.

This helps ensure that the model generalizes well and does not simply memorize the training data. I will use a test size of 20% (0.2) and `random_state` for reproducibility.

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (80, 10)
X_test shape: (20, 10)
y_train shape: (80,)
y_test shape: (20,)


### Train a Random Forest Classifier

I will now train a Random Forest Classifier using the training data (`X_train` and `y_train`). A Random Forest is an ensemble learning method that builds multiple decision trees and merges them together to get a more accurate and stable prediction. It's often effective for various types of data and can handle categorical features once they are encoded.

In [10]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

# Identify categorical columns
categorical_cols = X_train.select_dtypes(include=['object']).columns

# Apply one-hot encoding to training and testing sets
X_train_encoded = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

# Ensure both train and test sets have the same columns after encoding
# This handles cases where a category might be present in test but not train, or vice versa
train_cols = set(X_train_encoded.columns)
test_cols = set(X_test_encoded.columns)

missing_in_test = list(train_cols - test_cols)
for c in missing_in_test:
    X_test_encoded[c] = 0

missing_in_train = list(test_cols - train_cols)
for c in missing_in_train:
    X_train_encoded[c] = 0

X_test_encoded = X_test_encoded[X_train_encoded.columns] # Ensure order is the same

# Initialize the Random Forest Classifier
# Setting a random_state for reproducibility
model = RandomForestClassifier(random_state=42)

# Train the model on the encoded training data
model.fit(X_train_encoded, y_train)

print("Random Forest Classifier trained successfully!")

Random Forest Classifier trained successfully!


### Predict and Evaluate Model Performance

After training the Random Forest Classifier, it's essential to evaluate its performance on the test set (`X_test_encoded`) to understand how well it generalizes to new, unseen data. I will make predictions and then calculate the accuracy score.

In [11]:
from sklearn.metrics import accuracy_score

# Make predictions on the encoded test set
y_pred = model.predict(X_test_encoded)

# Calculate the accuracy score
accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy on the test set: {accuracy:.4f}")

Model Accuracy on the test set: 1.0000


## Predict Spoilage for New Farmer Data

Now that our model is trained and evaluated, we can use it to predict spoilage for new, unseen farmer data. Please upload the new data file (e.g., an Excel file) using the cell below.

In [12]:
# Upload the new farmer data file
from google.colab import files

uploaded_new_data = files.upload()

Saving new_raksha_data.xlsx to new_raksha_data.xlsx


In [13]:
import pandas as pd
import io

# Assuming the new data is an Excel file and the filename is known or can be inferred
# For simplicity, let's assume one file was uploaded and get its name
new_data_filename = next(iter(uploaded_new_data))
df_new = pd.read_excel(io.BytesIO(uploaded_new_data[new_data_filename]))

print(f"New data '{new_data_filename}' loaded successfully. First 5 rows:")
display(df_new.head())

New data 'new_raksha_data.xlsx' loaded successfully. First 5 rows:


,Crop_Type,Temperature_C,Humidity_Percent,Days_After_Harvest,Storage_Available,Storage_Temperature_C,Exposure_To_Sun_After_Harvest,Packaging_Type,Crop_Shelf_Life_Days
0,Tomato,38,85,7,No,35,6,Open,7
1,Onion,26,55,2,Yes,10,1,Crate,60
2,Potato,34,75,5,No,32,4,Sack,30
3,Wheat,24,45,1,Yes,8,0,Cold_Box,180
4,Maize,39,88,8,No,36,7,Open,150


### Preprocess New Data for Prediction

Before making predictions, the new data (`df_new`) must undergo the same preprocessing steps as the original training data (`X_train`). This includes:

1.  **Identifying categorical columns**.
2.  **Applying one-hot encoding** to convert these categorical columns into numerical format.
3.  **Ensuring column consistency** with the `X_train_encoded` dataset that the model was trained on. This is crucial to prevent errors if the new data has different categories or a different column order.

In [16]:
import pandas as pd

# Identify categorical columns in the new data
categorical_cols_new = df_new.select_dtypes(include=['object']).columns

# Apply one-hot encoding to the new data
df_new_encoded = pd.get_dummies(df_new, columns=categorical_cols_new, drop_first=True)

# Ensure the columns in the new encoded data match X_train_encoded
# Get the columns from the training data that the model was trained on
train_model_cols = X_train_encoded.columns

# Add missing columns to df_new_encoded and fill with 0
missing_cols_in_new = set(train_model_cols) - set(df_new_encoded.columns)
for c in missing_cols_in_new:
    df_new_encoded[c] = 0

# Drop extra columns from df_new_encoded that were not in train_model_cols
extra_cols_in_new = set(df_new_encoded.columns) - set(train_model_cols)
df_new_encoded = df_new_encoded.drop(columns=list(extra_cols_in_new))

# Ensure the order of columns is the same as in train_model_cols
df_new_encoded = df_new_encoded[train_model_cols]

print("New data preprocessed and columns aligned with training data. First 5 rows of encoded new data:")
display(df_new_encoded.head())

New data preprocessed and columns aligned with training data. First 5 rows of encoded new data:


,Temperature_C,Humidity_Percent,Days_After_Harvest,Storage_Temperature_C,Exposure_To_Sun_After_Harvest,Crop_Shelf_Life_Days,Location_Village_B,Location_Village_C,Crop_Type_Onion,Crop_Type_Potato,Crop_Type_Tomato,Crop_Type_Wheat,Storage_Available_Yes,Packaging_Type_Crate,Packaging_Type_Open,Packaging_Type_Sack
0,38,85,7,35,6,7,0,0,False,False,True,False,False,False,True,False
1,26,55,2,10,1,60,0,0,True,False,False,False,True,True,False,False
2,34,75,5,32,4,30,0,0,False,True,False,False,False,False,False,True
3,24,45,1,8,0,180,0,0,False,False,False,True,True,False,False,False
4,39,88,8,36,7,150,0,0,False,False,False,False,False,False,True,False


### Make Predictions on New Data

Now that the new farmer data is preprocessed and formatted correctly, we can use our trained Random Forest Classifier to predict the spoilage status for each entry.

In [17]:
# Make predictions on the preprocessed new data
new_predictions = model.predict(df_new_encoded)

# Add predictions to the original new data DataFrame for easy viewing
df_new['Predicted_Spoilage'] = new_predictions

# Map 1/0 back to 'Yes'/'No' for better interpretability
df_new['Predicted_Spoilage'] = df_new['Predicted_Spoilage'].map({1: 'Yes', 0: 'No'})

print("Spoilage predictions for the new farmer data:")
display(df_new[['Crop_Type', 'Temperature_C', 'Humidity_Percent', 'Predicted_Spoilage']].head())

Spoilage predictions for the new farmer data:


,Crop_Type,Temperature_C,Humidity_Percent,Predicted_Spoilage
0,Tomato,38,85,Yes
1,Onion,26,55,No
2,Potato,34,75,Yes
3,Wheat,24,45,No
4,Maize,39,88,Yes


## Recommendations for Farmers based on Spoilage Predictions

Based on the predicted spoilage for the new data, here are some actionable recommendations to help farmers minimize losses:

### For Crops Predicted to Spoil (Predicted_Spoilage = 'Yes'):

1.  **Immediate Action**: Prioritize these crops for sale or consumption. Consider direct sales to local markets, food processing, or charities to avoid complete loss.
2.  **Verify Conditions**: Double-check the storage conditions, temperature, humidity, and exposure to sun for these specific batches. There might be localized issues contributing to the higher spoilage risk.
3.  **Short-Term Storage**: If immediate sale isn't possible, transfer to optimal short-term storage with controlled temperature and humidity, even if it's not ideal for long-term.
4.  **Harvesting Review**: For future harvests of these crop types under similar conditions, review harvesting practices to ensure crops are not damaged or over-ripe at collection.

### For Crops Predicted Not to Spoil (Predicted_Spoilage = 'No'):

1.  **Maintain Optimal Conditions**: Continue to monitor and maintain the current favorable storage conditions. These conditions are effective in preventing spoilage for these batches.
2.  **Regular Checks**: Even for low-risk crops, conduct routine checks for any early signs of spoilage that might have been missed by the model due to unforeseen circumstances.
3.  **Longer-Term Planning**: These crops can be planned for sale or processing at a later date, allowing for better market opportunities or inventory management.

### General Preventative Measures:

*   **Improve Storage Infrastructure**: Invest in better storage solutions, such as cold storage or improved ventilation, especially for crops and locations frequently identified with spoilage risk.
*   **Optimized Packaging**: Use appropriate packaging types that extend shelf life and protect crops from physical damage and environmental factors.
*   **Harvesting and Post-Harvest Handling**: Train farmers on best practices for harvesting, handling, and transportation to minimize damage and contamination.
*   **Data Collection**: Encourage consistent and accurate data collection on all relevant factors (temperature, humidity, storage type, etc.) to continuously improve the accuracy and insights of spoilage prediction models.

## Detailed Record-by-Record Recommendations and Causes

While Random Forest models are powerful for prediction, pinpointing the *exact cause* for each individual prediction can be complex. However, we can infer likely contributing factors based on the input features and general agricultural knowledge. Below, I will provide a record-by-record analysis, offering recommendations and discussing potential causes for the predicted spoilage status.

In [19]:
from IPython.display import display, Markdown

for index, row in df_new.iterrows():
    report_lines = []
    report_lines.append(f"### Record {index + 1}: {row['Crop_Type']} - Predicted Spoilage: {row['Predicted_Spoilage']}\n")

    report_lines.append(f"**Summary:** This {row['Crop_Type']} is predicted to **{row['Predicted_Spoilage'].lower()}**.\n")

    if row['Predicted_Spoilage'] == 'Yes':
        report_lines.append("**Why?** Based on the data, several factors likely contribute to the prediction of spoilage:\n")
        causes = []
        if row['Temperature_C'] > 30: # Heuristic for high temp
            causes.append(f"*   High ambient temperature (**{row['Temperature_C']}°C**), which is often above optimal for fresh produce.")
        if row['Humidity_Percent'] > 80: # Heuristic for high humidity
            causes.append(f"*   High humidity (**{row['Humidity_Percent']}%**), which can encourage mold growth and microbial activity.")
        if row['Days_After_Harvest'] > 5: # Heuristic for long post-harvest time
            causes.append(f"*   Extended time after harvest (**{row['Days_After_Harvest']} days**), which naturally reduces the remaining shelf life.")
        if row['Exposure_To_Sun_After_Harvest'] > 3: # Heuristic for high sun exposure
            causes.append(f"*   Significant exposure to sun after harvest (**{row['Exposure_To_Sun_After_Harvest']} hours**), leading to faster deterioration.")
        if row['Storage_Available'] == 'No':
            causes.append("*   Lack of proper storage facilities, which makes maintaining ideal conditions difficult.")
        if row['Storage_Temperature_C'] > 25: # Heuristic for high storage temp
            causes.append(f"*   High storage temperature (**{row['Storage_Temperature_C']}°C**), accelerating metabolic processes and spoilage.")

        if causes:
            report_lines.extend(causes)
        else:
            report_lines.append("*   Conditions suggest a high risk of spoilage, though specific critical factors are not immediately obvious from our general thresholds. Further investigation of unique interactions might be needed.\n")

        report_lines.append("\n**Recommendation:** Prioritize this {row['Crop_Type']} for immediate sale or processing. Review and adjust current handling and storage protocols to mitigate similar risks for future batches.\n")

    else: # Predicted_Spoilage == 'No'
        report_lines.append("**Why?** The conditions for this record appear favorable, likely preventing spoilage:\n")
        factors = []
        if row['Temperature_C'] < 25:
            factors.append(f"*   Favorable ambient temperature (**{row['Temperature_C']}°C**), helping to preserve freshness.")
        if row['Humidity_Percent'] < 70:
            factors.append(f"*   Moderate humidity (**{row['Humidity_Percent']}%**), which is suitable for this crop's storage.")
        if row['Days_After_Harvest'] <= 3:
            factors.append(f"*   Short time after harvest (**{row['Days_After_Harvest']} days**), indicating recent harvesting and higher freshness.")
        if row['Exposure_To_Sun_After_Harvest'] <= 1:
            factors.append(f"*   Minimal exposure to sun after harvest (**{row['Exposure_To_Sun_After_Harvest']} hours**), protecting the crop from rapid degradation.")
        if row['Storage_Available'] == 'Yes':
            factors.append("*   Availability of proper storage facilities, allowing for controlled conditions.")
        if row['Storage_Temperature_C'] < 20:
            factors.append(f"*   Appropriate storage temperature (**{row['Storage_Temperature_C']}°C**), optimal for extending shelf life.")

        if factors:
            report_lines.extend(factors)
        else:
            report_lines.append("*   Current conditions are well-aligned with non-spoilage. Continue monitoring for any unexpected changes.\n")

        report_lines.append("\n**Recommendation:** Continue with the current storage and handling practices for this {row['Crop_Type']}. Conduct routine checks to ensure quality is maintained over time.\n")

    display(Markdown("\n---\n".join(report_lines)))


### Record 1: Tomato - Predicted Spoilage: Yes

---
**Summary:** This Tomato is predicted to **yes**.

---
**Why?** Based on the data, several factors likely contribute to the prediction of spoilage:

---
*   High ambient temperature (**38°C**), which is often above optimal for fresh produce.
---
*   High humidity (**85%**), which can encourage mold growth and microbial activity.
---
*   Extended time after harvest (**7 days**), which naturally reduces the remaining shelf life.
---
*   Significant exposure to sun after harvest (**6 hours**), leading to faster deterioration.
---
*   Lack of proper storage facilities, which makes maintaining ideal conditions difficult.
---
*   High storage temperature (**35°C**), accelerating metabolic processes and spoilage.
---

**Recommendation:** Prioritize this {row['Crop_Type']} for immediate sale or processing. Review and adjust current handling and storage protocols to mitigate similar risks for future batches.


### Record 2: Onion - Predicted Spoilage: No

---
**Summary:** This Onion is predicted to **no**.

---
**Why?** The conditions for this record appear favorable, likely preventing spoilage:

---
*   Moderate humidity (**55%**), which is suitable for this crop's storage.
---
*   Short time after harvest (**2 days**), indicating recent harvesting and higher freshness.
---
*   Minimal exposure to sun after harvest (**1 hours**), protecting the crop from rapid degradation.
---
*   Availability of proper storage facilities, allowing for controlled conditions.
---
*   Appropriate storage temperature (**10°C**), optimal for extending shelf life.
---

**Recommendation:** Continue with the current storage and handling practices for this {row['Crop_Type']}. Conduct routine checks to ensure quality is maintained over time.


### Record 3: Potato - Predicted Spoilage: Yes

---
**Summary:** This Potato is predicted to **yes**.

---
**Why?** Based on the data, several factors likely contribute to the prediction of spoilage:

---
*   High ambient temperature (**34°C**), which is often above optimal for fresh produce.
---
*   Significant exposure to sun after harvest (**4 hours**), leading to faster deterioration.
---
*   Lack of proper storage facilities, which makes maintaining ideal conditions difficult.
---
*   High storage temperature (**32°C**), accelerating metabolic processes and spoilage.
---

**Recommendation:** Prioritize this {row['Crop_Type']} for immediate sale or processing. Review and adjust current handling and storage protocols to mitigate similar risks for future batches.


### Record 4: Wheat - Predicted Spoilage: No

---
**Summary:** This Wheat is predicted to **no**.

---
**Why?** The conditions for this record appear favorable, likely preventing spoilage:

---
*   Favorable ambient temperature (**24°C**), helping to preserve freshness.
---
*   Moderate humidity (**45%**), which is suitable for this crop's storage.
---
*   Short time after harvest (**1 days**), indicating recent harvesting and higher freshness.
---
*   Minimal exposure to sun after harvest (**0 hours**), protecting the crop from rapid degradation.
---
*   Availability of proper storage facilities, allowing for controlled conditions.
---
*   Appropriate storage temperature (**8°C**), optimal for extending shelf life.
---

**Recommendation:** Continue with the current storage and handling practices for this {row['Crop_Type']}. Conduct routine checks to ensure quality is maintained over time.


### Record 5: Maize - Predicted Spoilage: Yes

---
**Summary:** This Maize is predicted to **yes**.

---
**Why?** Based on the data, several factors likely contribute to the prediction of spoilage:

---
*   High ambient temperature (**39°C**), which is often above optimal for fresh produce.
---
*   High humidity (**88%**), which can encourage mold growth and microbial activity.
---
*   Extended time after harvest (**8 days**), which naturally reduces the remaining shelf life.
---
*   Significant exposure to sun after harvest (**7 hours**), leading to faster deterioration.
---
*   Lack of proper storage facilities, which makes maintaining ideal conditions difficult.
---
*   High storage temperature (**36°C**), accelerating metabolic processes and spoilage.
---

**Recommendation:** Prioritize this {row['Crop_Type']} for immediate sale or processing. Review and adjust current handling and storage protocols to mitigate similar risks for future batches.


### Preprocess New Data for Prediction

Before making predictions, the new data (`df_new`) must undergo the same preprocessing steps as the original training data (`X_train`). This includes:

1.  **Identifying categorical columns**.
2.  **Applying one-hot encoding** to convert these categorical columns into numerical format.
3.  **Ensuring column consistency** with the `X_train_encoded` dataset that the model was trained on. This is crucial to prevent errors if the new data has different categories or a different column order.

In [14]:
# Identify categorical columns in the new data
categorical_cols_new = df_new.select_dtypes(include=['object']).columns

# Apply one-hot encoding to the new data
df_new_encoded = pd.get_dummies(df_new, columns=categorical_cols_new, drop_first=True)

# Ensure the columns in the new encoded data match X_train_encoded
# Get the columns from the training data that the model was trained on
train_model_cols = X_train_encoded.columns

# Add missing columns to df_new_encoded and fill with 0
missing_cols_in_new = set(train_model_cols) - set(df_new_encoded.columns)
for c in missing_cols_in_new:
    df_new_encoded[c] = 0

# Drop extra columns from df_new_encoded that were not in train_model_cols
extra_cols_in_new = set(df_new_encoded.columns) - set(train_model_cols)
df_new_encoded = df_new_encoded.drop(columns=list(extra_cols_in_new))

# Ensure the order of columns is the same as in train_model_cols
df_new_encoded = df_new_encoded[train_model_cols]

print("New data preprocessed and columns aligned with training data. First 5 rows of encoded new data:")
display(df_new_encoded.head())

New data preprocessed and columns aligned with training data. First 5 rows of encoded new data:


,Temperature_C,Humidity_Percent,Days_After_Harvest,Storage_Temperature_C,Exposure_To_Sun_After_Harvest,Crop_Shelf_Life_Days,Location_Village_B,Location_Village_C,Crop_Type_Onion,Crop_Type_Potato,Crop_Type_Tomato,Crop_Type_Wheat,Storage_Available_Yes,Packaging_Type_Crate,Packaging_Type_Open,Packaging_Type_Sack
0,38,85,7,35,6,7,0,0,False,False,True,False,False,False,True,False
1,26,55,2,10,1,60,0,0,True,False,False,False,True,True,False,False
2,34,75,5,32,4,30,0,0,False,True,False,False,False,False,False,True
3,24,45,1,8,0,180,0,0,False,False,False,True,True,False,False,False
4,39,88,8,36,7,150,0,0,False,False,False,False,False,False,True,False


### Make Predictions on New Data

Now that the new farmer data is preprocessed and formatted correctly, we can use our trained Random Forest Classifier to predict the spoilage status for each entry.

In [18]:
# Make predictions on the preprocessed new data
new_predictions = model.predict(df_new_encoded)

# Add predictions to the original new data DataFrame for easy viewing
df_new['Predicted_Spoilage'] = new_predictions

# Map 1/0 back to 'Yes'/'No' for better interpretability
df_new['Predicted_Spoilage'] = df_new['Predicted_Spoilage'].map({1: 'Yes', 0: 'No'})

print("Spoilage predictions for the new farmer data:")
display(df_new[['Crop_Type', 'Temperature_C', 'Humidity_Percent', 'Predicted_Spoilage']].head())

Spoilage predictions for the new farmer data:


,Crop_Type,Temperature_C,Humidity_Percent,Predicted_Spoilage
0,Tomato,38,85,Yes
1,Onion,26,55,No
2,Potato,34,75,Yes
3,Wheat,24,45,No
4,Maize,39,88,Yes


In [21]:
# Get spoilage probabilities for the new data
# model.predict_proba returns probabilities for each class (0 and 1)
# We are interested in the probability of spoilage, which is for class 1
spoilage_probabilities = model.predict_proba(df_new_encoded)[:, 1]

df_new['Spoilage_Probability'] = spoilage_probabilities

print("Spoilage Probabilities added to the DataFrame:")
display(df_new[['Crop_Type', 'Predicted_Spoilage', 'Spoilage_Probability']].head())

Spoilage Probabilities added to the DataFrame:


,Crop_Type,Predicted_Spoilage,Spoilage_Probability
0,Tomato,Yes,0.99
1,Onion,No,0.00
2,Potato,Yes,0.93
3,Wheat,No,0.02
4,Maize,Yes,1.00


In [22]:
# Define thresholds for spoilage levels
# These thresholds can be adjusted based on business needs and risk tolerance
low_threshold = 0.3
medium_threshold = 0.6

# Function to categorize spoilage level
def categorize_spoilage_level(prob):
    if prob < low_threshold:
        return 'Low'
    elif prob < medium_threshold:
        return 'Medium'
    else:
        return 'High'

# Apply the categorization function to create a new 'Spoilage_Level' column
df_new['Spoilage_Level'] = df_new['Spoilage_Probability'].apply(categorize_spoilage_level)

print("Spoilage levels (Low, Medium, High) added to the DataFrame:")
display(df_new[['Crop_Type', 'Predicted_Spoilage', 'Spoilage_Probability', 'Spoilage_Level']].head())

Spoilage levels (Low, Medium, High) added to the DataFrame:


,Crop_Type,Predicted_Spoilage,Spoilage_Probability,Spoilage_Level
0,Tomato,Yes,0.99,High
1,Onion,No,0.00,Low
2,Potato,Yes,0.93,High
3,Wheat,No,0.02,Low
4,Maize,Yes,1.00,High


In [23]:
import joblib

# Save the trained model
joblib.dump(model, 'random_forest_model.joblib')

print("Trained model saved as 'random_forest_model.joblib'")

Trained model saved as 'random_forest_model.joblib'


In [20]:
df_new.to_csv('new_raksha_data_with_predictions.csv', index=False)
print('DataFrame with predictions saved to new_raksha_data_with_predictions.csv')


DataFrame with predictions saved to new_raksha_data_with_predictions.csv
